# OTC Navigator: Invented Facet Development

## Objective

Develop a new facet called Documented Safety Risk
using the warning texts associated with OTC medicines.

The facet will be developed through an exploratory
review of warning texts, identification of recurring
risk concepts, and iterative refinement of category
definitions.

A stratified sample of 200 medicines will be manually
classified to establish ground truth for automatic
classification.

The classification represents risks documented in
the available warning excerpts, rather than a
comprehensive assessment of medication safety.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/content/OTC_Medicines_Standardized.csv"
)

print("Dataset dimensions:", df.shape)

print("\nTherapeutic group distribution:")
print(df["Groups"].value_counts())

Dataset dimensions: (300, 15)

Therapeutic group distribution:
Groups
Pain and Fever           60
Cough Cold & Flu         60
Allergy & Respiratory    60
Digestive & Heartburn    60
Skin & Topical           60
Name: count, dtype: int64


## Exploratory Analysis of Warning Texts

Before defining the invented facet, warning texts
and preliminary codes are examined across all five
therapeutic groups.

This exploratory analysis will help identify
recurring concepts, ambiguous cases, and potential
classification categories.

In [2]:
# Select 3 medicines from each therapeutic group
# for exploratory analysis

exploratory_sample = (
    df.groupby("Groups", group_keys=False)
    .sample(n=3, random_state=42)
    .sort_values(["Groups", "Item"])
)

# Display complete warning texts and initial codes

for _, row in exploratory_sample.iterrows():

    print("=" * 80)
    print("Item:", row["Item"])
    print("Group:", row["Groups"])
    print("Product:", row["Product"])

    print("\nSelected Warnings:")
    print(row["Selected Warnings"])

    print("\nInitial Codes:")
    print(row["Initial Codes"])

    print()

Item: 121
Group: Allergy & Respiratory
Product: Claritin 24 Hour Allergy

Selected Warnings:
Ask a doctor if you have liver or kidney disease; taking more 
than directed may cause drowsiness.

Initial Codes:
reducing fever; relieving cold or nasal symptoms; relieving allergy symptoms; relieving itching and irritation; warning against excessive acetaminophen use; highlighting sedation risks; highlighting kidney disease precautions

Item: 126
Group: Allergy & Respiratory
Product: Children's Zyrtec Allergy

Selected Warnings:
Drowsiness may occur and can increase with sedatives/alcohol; 
ask a doctor with liver/kidney disease.

Initial Codes:
relieving cold or nasal symptoms; relieving allergy symptoms; relieving itching and irritation; warning against excessive acetaminophen use; highlighting sedation risks; highlighting kidney disease precautions

Item: 157
Group: Allergy & Respiratory
Product: Children's Benadryl Allergy Liquid

Selected Warnings:
May cause marked drowsiness; do not us

## Development of the Invented Facet

Facet name: Documented Safety Risk

Classificatory principle:
The type of adverse health outcome explicitly
documented in the selected warning text.

Unit of classification:
Individual OTC medicine records.

Classification structure:
Multi-label classification.

Evidence source:
Selected Warnings.

Preliminary categories:
R01 - Organ toxicity
R02 - Bleeding
R03 - Cardiovascular effects
R04 - Allergic reactions
R05 - Neurological and behavioral effects
R06 - Skin and local tissue effects
R07 - Gastrointestinal effects
R08 - Other documented risk
R09 - Risk not specified

Classification rules:
1. Assign categories only when supported by
   explicit evidence in the warning text.
2. Multiple risk categories may apply to a medicine.
3. Avoid inferring adverse outcomes from precautions.
4. Use R08 for explicitly documented risks that
   do not fit the existing categories.
5. Use R09 when no specific adverse outcome is stated.
6. R09 cannot coexist with other risk categories.
7. Missing warning text must be treated separately.
8. Initial Codes are not used as ground truth.

The absence of a risk category indicates that the
risk is not documented in the available excerpt,
not that the medicine is free from that risk.

The preliminary codebook will be refined through
pilot classification.

## Stratified Sampling for Manual Classification

A stratified random sample of 200 medicines is
selected from the complete dataset of 300 medicines.

Therapeutic group is used as the stratification
variable, with 40 medicines randomly selected
from each of the five groups.

This ensures proportional representation of all
therapeutic groups in the manually classified dataset.

A fixed random seed is used for reproducibility.

The remaining 100 medicines are reserved for
automatic classification after model evaluation.

In [3]:
# Generate stratified sample of 200 medicines

manual_sample = (
    df.groupby("Groups", group_keys=False)
    .sample(n=40, random_state=42)
    .sort_values("Item")
    .reset_index(drop=True)
)

# Identify the remaining 100 medicines

unlabeled_sample = (
    df.loc[~df["Item"].isin(manual_sample["Item"])]
    .sort_values("Item")
    .reset_index(drop=True)
)

# Check sample sizes

print("Complete dataset:", len(df))
print("Manual classification sample:", len(manual_sample))
print("Unlabeled sample:", len(unlabeled_sample))

# Check representation of therapeutic groups

print("\nManual classification sample distribution:")
print(manual_sample["Groups"].value_counts())

print("\nUnlabeled sample distribution:")
print(unlabeled_sample["Groups"].value_counts())

# Verify that there is no overlap

overlap = set(manual_sample["Item"]).intersection(
    unlabeled_sample["Item"]
)

print("\nOverlapping item IDs:", len(overlap))

Complete dataset: 300
Manual classification sample: 200
Unlabeled sample: 100

Manual classification sample distribution:
Groups
Pain and Fever           40
Cough Cold & Flu         40
Allergy & Respiratory    40
Digestive & Heartburn    40
Skin & Topical           40
Name: count, dtype: int64

Unlabeled sample distribution:
Groups
Pain and Fever           20
Cough Cold & Flu         20
Allergy & Respiratory    20
Digestive & Heartburn    20
Skin & Topical           20
Name: count, dtype: int64

Overlapping item IDs: 0


## Pilot Classification

A pilot sample of 20 medicines is selected from
the stratified manual classification dataset.

Four medicines are randomly selected from each
therapeutic group.

The purpose of pilot classification is to:
- Evaluate the clarity of category definitions.
- Identify overlapping or ambiguous categories.
- Identify risks not covered by the codebook.
- Refine inclusion and exclusion criteria.
- Establish consistent classification rules.

The pilot sample is part of the 200-item manual
classification dataset.

In [4]:
# Select 4 medicines from each therapeutic group

pilot_sample = (
    manual_sample.groupby("Groups", group_keys=False)
    .sample(n=4, random_state=42)
    .sort_values("Item")
    .reset_index(drop=True)
)

print("Pilot sample size:", len(pilot_sample))

print("\nPilot sample distribution:")
print(pilot_sample["Groups"].value_counts())

display(
    pilot_sample[["Item", "Groups", "Product"]]
)

Pilot sample size: 20

Pilot sample distribution:
Groups
Pain and Fever           4
Cough Cold & Flu         4
Allergy & Respiratory    4
Digestive & Heartburn    4
Skin & Topical           4
Name: count, dtype: int64


,Item,Groups,Product
0,38,Pain and Fever,Bufferin Aspirin
1,43,Pain and Fever,Aspercreme Original Pain Relief Cream
2,47,Pain and Fever,Advil PM
3,54,Pain and Fever,Motrin Junior Strength Chewables
4,61,Cough Cold & Flu,Vicks DayQuil Cold & Flu
5,71,Cough Cold & Flu,Mucinex Extended-Release
6,97,Cough Cold & Flu,Vicks DayQuil Cough
7,103,Cough Cold & Flu,Alka-Seltzer Plus Severe Cold & Flu
8,146,Allergy & Respiratory,Allegra-D 12 Hour
9,147,Allergy & Respiratory,Benadryl Allergy Liqui-Gels


In [5]:
# Select relevant columns for manual classification

pilot_template = pilot_sample[
    ["Item", "Groups", "Product", "Selected Warnings"]
].copy()

# Define classification columns

risk_columns = [
    "R01_Organ_Toxicity",
    "R02_Bleeding",
    "R03_Cardiovascular",
    "R04_Allergic_Reactions",
    "R05_Neurological_Behavioral",
    "R06_Skin_Local_Tissue",
    "R07_Gastrointestinal",
    "R08_Other_Documented_Risk",
    "R09_Risk_Not_Specified"
]

# Create empty columns for manual labeling

for col in risk_columns:
    pilot_template[col] = ""

# Additional columns for qualitative observations

pilot_template["Evidence"] = ""
pilot_template["Coding_Notes"] = ""

# Export to Excel

pilot_template.to_excel(
    "OTC_Pilot_Classification.xlsx",
    index=False
)

print("Pilot classification template created!")
print("Dimensions:", pilot_template.shape)

Pilot classification template created!
Dimensions: (20, 15)


In [6]:
from google.colab import files

files.download("OTC_Pilot_Classification.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Finalization of the Classification Codebook

Following pilot classification, the nine preliminary
risk categories are retained.

The following classification rules are established:

1. Classify explicitly documented adverse outcomes.
2. Allow multiple risk categories per medicine.
3. Do not infer harm from pre-existing conditions,
   drug interactions, or usage restrictions.
4. Classify sensory adverse effects, such as bitter
   taste, under R08.
5. Classify named adverse syndromes not covered by
   existing categories under R08.
6. Symptoms requiring medical consultation are not
   classified as adverse effects unless the warning
   explicitly associates them with medication use.
7. Assign R09 only when no specific adverse outcome
   is documented in an available warning excerpt.
8. R09 cannot coexist with R01–R08.

All classifications describe documented information,
not the complete safety profile of a medicine.

## Validation of Pilot Classification

The completed pilot classification is imported
and checked for completeness, valid binary labels,
category coverage, and classification consistency.

In [8]:
pilot_labeled = pd.read_excel(
    "/content/OTC_Pilot_Classification Final.xlsx"
)

display(pilot_labeled.head())

,Item,Groups,Product,Selected Warnings,R01_Organ_Toxicity,R02_Bleeding,R03_Cardiovascular,R04_Allergic_Reactions,R05_Neurological_Behavioral,R06_Skin_Local_Tissue,R07_Gastrointestinal,R08_Other_Documented_Risk,R09_Risk_Not_Specified,Evidence,Coding_Notes
0,38,Pain and Fever,Bufferin Aspirin,Aspirin may cause severe stomach bleeding; avo...,0,1,0,0,0,0,0,0,0,Severe stomach bleeding,Explicit bleeding risk
1,43,Pain and Fever,Aspercreme Original Pain Relief Cream,External use only; avoid eyes and broken skin.,0,0,0,0,0,0,0,0,1,Avoid eyes and broken skin,Application restriction only
2,47,Pain and Fever,Advil PM,NSAID may cause stomach bleeding; antihistamin...,0,1,0,0,1,0,0,0,0,"Stomach bleeding, marked drowsiness",2 distinct adverse outcomes
3,54,Pain and Fever,Motrin Junior Strength Chewables,NSAIDs may cause stomach bleeding or severe al...,0,1,0,1,0,0,0,0,0,"Stomach bleeding, severe allergic reactions",2 distinct adverse outcomes
4,61,Cough Cold & Flu,Vicks DayQuil Cold & Flu,Acetaminophen can cause severe liver damage if...,1,0,0,0,0,0,0,0,0,Severe liver damage,Explicit organ toxicity


In [9]:
# Extract the nine risk classification columns
labels = pilot_labeled[risk_columns]

print("Total pilot medicines:", len(pilot_labeled))

# Check for missing classifications
print("\nMissing classification values:")
print(labels.isnull().sum())

# Check that all values are either 0 or 1
print("\nInvalid classification values:",
      (~labels.isin([0, 1])).sum().sum())

# Calculate risk category frequencies
print("\nRisk category frequencies:")
print(labels.sum())

# Check R09 exclusivity
conflicts = pilot_labeled[
    (labels[risk_columns[-1]] == 1) &
    (labels[risk_columns[:-1]].sum(axis=1) > 0)
]

print("\nR09 classification conflicts:", len(conflicts))

# Check for medicines without any assigned category
print("Rows without any category:",
      (labels.sum(axis=1) == 0).sum())

# Verify that the correct pilot items were labeled
print("\nCorrect pilot item IDs:",
      sorted(pilot_labeled["Item"].tolist()) ==
      sorted(pilot_sample["Item"].tolist()))

Total pilot medicines: 20

Missing classification values:
R01_Organ_Toxicity             0
R02_Bleeding                   0
R03_Cardiovascular             0
R04_Allergic_Reactions         0
R05_Neurological_Behavioral    0
R06_Skin_Local_Tissue          0
R07_Gastrointestinal           0
R08_Other_Documented_Risk      0
R09_Risk_Not_Specified         0
dtype: int64

Invalid classification values: 0

Risk category frequencies:
R01_Organ_Toxicity              1
R02_Bleeding                    3
R03_Cardiovascular              1
R04_Allergic_Reactions          1
R05_Neurological_Behavioral     5
R06_Skin_Local_Tissue           0
R07_Gastrointestinal            0
R08_Other_Documented_Risk       2
R09_Risk_Not_Specified         11
dtype: int64

R09 classification conflicts: 0
Rows without any category: 0

Correct pilot item IDs: True


## Preparation of the Remaining Manual Classification Sample

Following pilot classification and structural
validation, the remaining 180 medicines are
prepared for manual classification using the
finalized codebook.

The 20 pilot medicines are excluded from this
sample to avoid duplicate classification.

Each therapeutic group contributes 36 additional
medicines.

The completed classifications will be combined
with the pilot classifications to create the
200-item labeled dataset.

In [10]:
# Exclude the 20 completed pilot medicines

remaining_sample = (
    manual_sample.loc[
        ~manual_sample["Item"].isin(pilot_labeled["Item"])
    ]
    .sort_values("Item")
    .reset_index(drop=True)
)

# Verify sample size and distribution

print("Remaining medicines:", len(remaining_sample))

print("\nTherapeutic group distribution:")
print(remaining_sample["Groups"].value_counts())

print("\nOverlap with pilot sample:",
      len(set(remaining_sample["Item"]) &
          set(pilot_labeled["Item"])))

Remaining medicines: 180

Therapeutic group distribution:
Groups
Pain and Fever           36
Cough Cold & Flu         36
Allergy & Respiratory    36
Digestive & Heartburn    36
Skin & Topical           36
Name: count, dtype: int64

Overlap with pilot sample: 0


In [11]:
# Select relevant columns

remaining_template = remaining_sample[
    ["Item", "Groups", "Product", "Selected Warnings"]
].copy()

# Add empty risk classification columns

for col in risk_columns:
    remaining_template[col] = ""

# Add qualitative observation columns

remaining_template["Evidence"] = ""
remaining_template["Coding_Notes"] = ""

# Export the template

remaining_template.to_excel(
    "OTC_Remaining_180_Classification.xlsx",
    index=False
)

print("Classification template created!")
print("Dimensions:", remaining_template.shape)

Classification template created!
Dimensions: (180, 15)


In [12]:
from google.colab import files

files.download(
    "OTC_Remaining_180_Classification.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Simplified Manual Classification

To improve coding efficiency, a single-column
classification template is used to record
applicable risk codes for each medicine.

Multiple risk codes are separated by semicolons.
The assigned codes will subsequently be converted
into binary labels for machine learning.

This approach preserves the multi-label
classification structure of the original taxonomy.

In [13]:
# Create a simplified classification template

simple_template = remaining_sample[
    ["Item", "Groups", "Product", "Selected Warnings"]
].copy()

# Add classification columns

simple_template["Risk_Codes"] = ""
simple_template["Evidence"] = ""
simple_template["Coding_Notes"] = ""

# Export the simplified template

simple_template.to_excel(
    "OTC_Simplified_180_Classification.xlsx",
    index=False
)

print("Simplified template created!")
print("Dimensions:", simple_template.shape)

display(simple_template.head())

Simplified template created!
Dimensions: (180, 7)


,Item,Groups,Product,Selected Warnings,Risk_Codes,Evidence,Coding_Notes
0,2,Pain and Fever,Advil,NSAID may cause severe stomach bleeding; may i...,,,
1,4,Pain and Fever,Bayer Genuine Aspirin,Children/teens recovering from chickenpox or f...,,,
2,5,Pain and Fever,Children's Tylenol,Severe liver damage possible if child exceeds ...,,,
3,7,Pain and Fever,Children's Motrin,NSAID may cause severe stomach bleeding; NSAID...,,,
4,10,Pain and Fever,Equate Pain Reliever,Severe liver damage possible with excessive \n...,,,


In [14]:
from google.colab import files

files.download("OTC_Simplified_180_Classification.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Import and Validation of Completed Classifications

The 180 human-verified, AI-assisted classifications
are imported and validated against the finalized
Documented Safety Risk codebook.

Validation includes sample coverage, valid risk
codes, missing classifications, duplicate item IDs,
and exclusivity of the residual category (R09).

In [15]:
completed_180 = pd.read_excel(
    "/content/OTC Simplified Classification final.xlsx"
)

print("Dataset dimensions:", completed_180.shape)

display(completed_180.head())

Dataset dimensions: (180, 7)


,Item,Groups,Product,Selected Warnings,Risk_Codes,Evidence,Coding_Notes
0,2,Pain and Fever,Advil,NSAID may cause severe stomach bleeding; may i...,R02; R03,NSAID may cause severe stomach bleeding; may i...,Explicit outcome(s) in excerpt; no additional ...
1,4,Pain and Fever,Bayer Genuine Aspirin,Children/teens recovering from chickenpox or f...,R02; R08,Children/teens recovering from chickenpox or f...,Explicit outcome(s) in excerpt; no additional ...
2,5,Pain and Fever,Children's Tylenol,Severe liver damage possible if child exceeds ...,R01; R06,Severe liver damage possible if child exceeds ...,Explicit outcome(s) in excerpt; no additional ...
3,7,Pain and Fever,Children's Motrin,NSAID may cause severe stomach bleeding; NSAID...,R02; R03,NSAID may cause severe stomach bleeding; NSAID...,Explicit outcome(s) in excerpt; no additional ...
4,10,Pain and Fever,Equate Pain Reliever,Severe liver damage possible with excessive \n...,R01; R06,Severe liver damage possible with excessive us...,Explicit outcome(s) in excerpt; no additional ...


In [16]:
# Define valid risk codes

valid_codes = {f"R{i:02d}" for i in range(1, 10)}

# Convert semicolon-separated codes into lists

completed_180["Parsed_Codes"] = (
    completed_180["Risk_Codes"]
    .fillna("")
    .astype(str)
    .apply(lambda x: [
        code.strip().upper()
        for code in x.split(";")
        if code.strip()
    ])
)

# Identify invalid risk codes

invalid_rows = completed_180[
    completed_180["Parsed_Codes"].apply(
        lambda codes: not set(codes).issubset(valid_codes)
    )
]

# Identify missing classifications

missing_rows = completed_180[
    completed_180["Parsed_Codes"].apply(len) == 0
]

# Check R09 exclusivity

conflict_rows = completed_180[
    completed_180["Parsed_Codes"].apply(
        lambda codes: "R09" in codes and len(set(codes)) > 1
    )
]

# Print validation results

print("Total medicines:", len(completed_180))

print("Duplicate item IDs:",
      completed_180["Item"].duplicated().sum())

print("Missing classifications:", len(missing_rows))

print("Invalid risk codes:", len(invalid_rows))

print("R09 conflicts:", len(conflict_rows))

print("Correct sample IDs:",
      sorted(completed_180["Item"].tolist()) ==
      sorted(remaining_sample["Item"].tolist()))

print("Overlap with pilot:",
      len(set(completed_180["Item"]) &
          set(pilot_labeled["Item"])))

Total medicines: 180
Duplicate item IDs: 0
Missing classifications: 0
Invalid risk codes: 0
R09 conflicts: 0
Correct sample IDs: True
Overlap with pilot: 0


## Binary Encoding of Risk Categories

The semicolon-separated risk codes are converted
into binary indicator columns using multi-label
encoding.

Each risk category is represented by:

- 1: The risk is explicitly documented.
- 0: The risk is not documented in the selected
  warning excerpt.

This encoding prepares the dataset for multi-label
machine learning classification while preserving
the original taxonomy.

In [17]:
# Convert risk codes into binary columns

for col in risk_columns:

    code = col.split("_")[0]

    completed_180[col] = completed_180["Parsed_Codes"].apply(
        lambda codes: int(code in codes)
    )

# Verify the binary encoding

display(
    completed_180[
        ["Item", "Product", "Risk_Codes"] + risk_columns
    ].head(10)
)

print("\nRisk category frequencies:")
print(completed_180[risk_columns].sum())

,Item,Product,Risk_Codes,R01_Organ_Toxicity,R02_Bleeding,R03_Cardiovascular,R04_Allergic_Reactions,R05_Neurological_Behavioral,R06_Skin_Local_Tissue,R07_Gastrointestinal,R08_Other_Documented_Risk,R09_Risk_Not_Specified
0,2,Advil,R02; R03,0,1,1,0,0,0,0,0,0
1,4,Bayer Genuine Aspirin,R02; R08,0,1,0,0,0,0,0,1,0
2,5,Children's Tylenol,R01; R06,1,0,0,0,0,1,0,0,0
3,7,Children's Motrin,R02; R03,0,1,1,0,0,0,0,0,0
4,10,Equate Pain Reliever,R01; R06,1,0,0,0,0,1,0,0,0
5,12,Tylenol Regular Strength,R01; R06,1,0,0,0,0,1,0,0,0
6,14,Advil Liqui-Gels,R02; R03,0,1,1,0,0,0,0,0,0
7,15,Advil Dual Action with Acetaminophen,R01; R02; R03,1,1,1,0,0,0,0,0,0
8,16,Advil Migraine,R02; R03,0,1,1,0,0,0,0,0,0
9,17,Motrin IB,R02; R03,0,1,1,0,0,0,0,0,0



Risk category frequencies:
R01_Organ_Toxicity             19
R02_Bleeding                   25
R03_Cardiovascular             15
R04_Allergic_Reactions          2
R05_Neurological_Behavioral    34
R06_Skin_Local_Tissue          19
R07_Gastrointestinal            0
R08_Other_Documented_Risk      18
R09_Risk_Not_Specified         90
dtype: int64


## Construction of the Labeled Dataset

The 20 pilot classifications and 180 subsequent
classifications are combined into a stratified
200-item labeled dataset.

The dataset contains 40 medicines from each of
the five therapeutic groups.

Risk labels were generated with AI assistance
and subsequently reviewed by a human.

The remaining 100 medicines are reserved for
automatic classification after model development
and evaluation.

In [18]:
# Define common columns

common_columns = [
    "Item",
    "Groups",
    "Product",
    "Selected Warnings"
] + risk_columns + [
    "Evidence",
    "Coding_Notes"
]

# Combine pilot and remaining classifications

labeled_200 = pd.concat(
    [
        pilot_labeled[common_columns],
        completed_180[common_columns]
    ],
    ignore_index=True
).sort_values("Item").reset_index(drop=True)

# Validate combined dataset

print("Dataset dimensions:", labeled_200.shape)

print("\nTherapeutic group distribution:")
print(labeled_200["Groups"].value_counts())

print("\nDuplicate item IDs:",
      labeled_200["Item"].duplicated().sum())

print("Missing risk labels:",
      labeled_200[risk_columns].isnull().sum().sum())

print("\nRisk category frequencies:")
print(labeled_200[risk_columns].sum())

print("\nCorrect sample IDs:",
      sorted(labeled_200["Item"].tolist()) ==
      sorted(manual_sample["Item"].tolist()))

Dataset dimensions: (200, 15)

Therapeutic group distribution:
Groups
Pain and Fever           40
Cough Cold & Flu         40
Allergy & Respiratory    40
Digestive & Heartburn    40
Skin & Topical           40
Name: count, dtype: int64

Duplicate item IDs: 0
Missing risk labels: 0

Risk category frequencies:
R01_Organ_Toxicity              20
R02_Bleeding                    28
R03_Cardiovascular              16
R04_Allergic_Reactions           3
R05_Neurological_Behavioral     39
R06_Skin_Local_Tissue           19
R07_Gastrointestinal             0
R08_Other_Documented_Risk       20
R09_Risk_Not_Specified         101
dtype: int64

Correct sample IDs: True


## Final Labeled Dataset and Export

The completed dataset contains 200 medicines,
with 40 medicines from each therapeutic group.

The Documented Safety Risk facet contains nine
categories represented through multi-label
binary encoding.

The classification process involved AI-assisted
label generation followed by human verification.

The dataset exhibits substantial class imbalance,
including rare and unrepresented categories.

These limitations will be considered during
automatic classification and model evaluation.

The labeled dataset and the remaining 100
unlabeled medicines are exported for the next
stage of the project.

In [19]:
# Final structural validation

final_labels = labeled_200[risk_columns]

assert len(labeled_200) == 200
assert labeled_200["Item"].is_unique
assert final_labels.isin([0, 1]).all().all()
assert (final_labels.sum(axis=1) > 0).all()

assert not (
    (final_labels[risk_columns[-1]] == 1) &
    (final_labels[risk_columns[:-1]].sum(axis=1) > 0)
).any()

assert len(unlabeled_sample) == 100

assert set(labeled_200["Item"]).isdisjoint(
    unlabeled_sample["Item"]
)

assert (
    set(labeled_200["Item"]) |
    set(unlabeled_sample["Item"])
) == set(df["Item"])

# Export the final labeled dataset

labeled_200.to_csv(
    "OTC_Labeled_200.csv",
    index=False
)

# Export the remaining unlabeled medicines

unlabeled_sample.to_csv(
    "OTC_Unlabeled_100.csv",
    index=False
)

# Export category distribution

risk_summary = pd.DataFrame({
    "Risk_Category": risk_columns,
    "Count": labeled_200[risk_columns].sum().values
})

risk_summary["Percentage"] = (
    risk_summary["Count"] / len(labeled_200) * 100
)

risk_summary.to_csv(
    "OTC_Risk_Category_Distribution.csv",
    index=False
)

print("Final validation passed!")
print("All three files exported successfully.")

display(risk_summary)

Final validation passed!
All three files exported successfully.


,Risk_Category,Count,Percentage
0,R01_Organ_Toxicity,20,10.0
1,R02_Bleeding,28,14.0
2,R03_Cardiovascular,16,8.0
3,R04_Allergic_Reactions,3,1.5
4,R05_Neurological_Behavioral,39,19.5
5,R06_Skin_Local_Tissue,19,9.5
6,R07_Gastrointestinal,0,0.0
7,R08_Other_Documented_Risk,20,10.0
8,R09_Risk_Not_Specified,101,50.5


In [20]:
from google.colab import files

files.download("OTC_Labeled_200.csv")
files.download("OTC_Unlabeled_100.csv")
files.download("OTC_Risk_Category_Distribution.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
import zipfile
from google.colab import files

with zipfile.ZipFile("OTC_Classification_Data.zip", "w") as zipf:
    zipf.write("OTC_Labeled_200.csv")
    zipf.write("OTC_Unlabeled_100.csv")
    zipf.write("OTC_Risk_Category_Distribution.csv")

files.download("OTC_Classification_Data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>